In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from gensim.models import FastText
from nltk.tokenize import word_tokenize
from dataset_class.job_post_dataset import JobPostingDataset
from model_construction.model import FakeJobDetector
import os, json, pickle
import warnings
warnings.filterwarnings("ignore", message=".*cuBLAS.*")


In [2]:
# Load data
combined_df = pd.read_csv("./data/clean/fake_job_postings_ALL.csv")

numeric_cols = ["telecommuting", "missing_count", "total_text_len", "company_profile_len", "description_len", 
                "requirements_len", "company_profile_word_count", "description_word_count", 
                "requirements_word_count", "salary_provided", "has_company_profile",
                "vague_location", "has_company_logo", "has_questions","benefits_len","benefits_word_count",]


# Same split as training
train_data, temp_data = train_test_split(
    combined_df, test_size=0.2, random_state=42, stratify=combined_df['fraudulent']
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

X_test_text    = test_data['full_text'].tolist()
X_test_numeric = test_data[numeric_cols].values.astype(np.float32)
y_test         = test_data['fraudulent'].values.tolist()

# Scale numeric features using training data (same as training)
X_train_numeric = train_data[numeric_cols].values.astype(np.float32)
non_binary_cols = [
    "missing_count", "total_text_len", 
    "company_profile_len", "description_len", "requirements_len", "benefits_len",
    "company_profile_word_count", "description_word_count", "requirements_word_count", 
    "benefits_word_count"
]

non_binary_indices = [numeric_cols.index(col) for col in non_binary_cols]

scaler = StandardScaler()
scaler.fit(X_train_numeric[:, non_binary_indices])
X_test_numeric[:, non_binary_indices] = scaler.transform(X_test_numeric[:, non_binary_indices])

print(f"Test samples: {len(y_test)}, Fakes: {sum(y_test)}")

Test samples: 1788, Fakes: 86


In [3]:
# Load FastText and build vocab inline (matches Model_training.ipynb)
fasttext_model = FastText.load("./optimal_fasttext.bin")

vocab = {"<unk>": 0, "<pad>": 1}
for idx, word in enumerate(fasttext_model.wv.index_to_key):
    vocab[word] = idx + 2

# Build reverse vocab (id -> word) for later
id_to_word = {idx: word for word, idx in vocab.items()}

# Tokenize test text
test_tk = [word_tokenize(text.lower()) for text in X_test_text]

def encode(tokens, vocab):
    return [vocab.get(t, 0) for t in tokens]

test_texts_tok = [encode(s, vocab) for s in test_tk]

# Create test dataset
MAX_LEN = 2048
test_dataset = JobPostingDataset(test_texts_tok, X_test_numeric, y_test, max_len=MAX_LEN)

print(f"Vocab size: {len(vocab)}, FastText dim: {fasttext_model.wv.vector_size}")

Vocab size: 26973, FastText dim: 100


In [4]:
# Build embedding matrix and load model
vocab_size = len(vocab)
embed_dim = fasttext_model.wv.vector_size

embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)
for word, idx in vocab.items():
    if word in fasttext_model.wv:
        embedding_matrix[idx] = fasttext_model.wv[word]

pretrained_embeddings = torch.tensor(embedding_matrix, dtype=torch.float)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FakeJobDetector(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    gru_hidden_dim=64,
    num_numerical_features=X_test_numeric.shape[1],
    num_hidden_dim=128,
    pretrained_embeddings=pretrained_embeddings,
    device=device,
)
model.load("best_model.pt")

# Freeze everything, model weights, embeddings, all parameters
model.eval()
for param in model.parameters():
    param.requires_grad = False

print("Model loaded and fully frozen")
print(f"Device: {device}")


Model loaded and fully frozen
Device: cuda


In [5]:
# Find a fake sample correctly classified as fake
threshold = 0.3

for i in range(len(test_dataset)):
    inputs, label = test_dataset[i]
    if label == 1:  # fake 
        input_ids = inputs['input_ids'].unsqueeze(0).to(device)
        attention_mask = inputs['attention_mask'].unsqueeze(0).to(device)
        numerical_features = inputs['numerical_features'].unsqueeze(0).to(device)
        
        with torch.no_grad():
            logit = model(input_ids, attention_mask, numerical_features)
            prob = torch.sigmoid(logit).item()
        
        if prob >= threshold:  # model correctly says fake
            sample_idx = i
            original_prob = prob
            # Get the original words
            original_tokens = test_texts_tok[i]
            original_words = [id_to_word.get(t, "<unk>") for t in original_tokens]
            print(f"Sample index: {i}")
            print(f"P(fake) = {original_prob:.4f}")
            print(f"Text preview: {' '.join(original_words[:])}")
            break


Sample index: 45
P(fake) = 0.9145
Text preview: nurse shift program supervisor us ky lexington apply using below linklink nurse shift program supervisor on one of five nursing units at thomson hood veterans center , a state owned long term , 285 bed , nursing care facility for kentucky s veterans and is located in jessamine county . you may contact <unk> <unk> at 1 link submit applications online at link apply to req <unk> br thomson hood veterans center does not discriminate against any person on the basis of race , color , national origin , disability , or age in admission , treatment , or participation in its programs , services , and activities , or in employment . for further information about this policy , contact amy lambert , telephone number tdd full time hospital health care


In [6]:
# FGSM Attack
epsilon = 0.3

inputs, label = test_dataset[sample_idx]
input_ids = inputs['input_ids'].unsqueeze(0).to(device)
attention_mask = inputs['attention_mask'].unsqueeze(0).to(device)
numerical_features = inputs['numerical_features'].unsqueeze(0).to(device)

# Must be in train mode for GRU backward, but weights are still frozen
model.train()

# Get embeddings and enable gradient on them
embedded = model.embedding(input_ids)
embedded = embedded.detach().requires_grad_(True)

# Forward pass bypassing embedding layer
gru_out = model.bigru(embedded)
nlp_out = model.attention(gru_out, attention_mask.to(device))
num_out = model.numerical(numerical_features)
merged = torch.cat([nlp_out, num_out], dim=1)
logit = model.classifier(merged).squeeze(-1)

# Compute gradient
logit.backward()

# FGSM perturbation
perturbed = embedded.detach() - epsilon * embedded.grad.sign()

# Forward with perturbed embeddings
model.eval()
with torch.no_grad():
    gru_out2 = model.bigru(perturbed)
    nlp_out2 = model.attention(gru_out2, attention_mask.to(device))
    num_out2 = model.numerical(numerical_features)
    merged2 = torch.cat([nlp_out2, num_out2], dim=1)
    logit2 = model.classifier(merged2).squeeze(-1)
    new_prob = torch.sigmoid(logit2).item()

print(f"Before FGSM: P(fake) = {original_prob:.4f}")
print(f"After FGSM:  P(fake) = {new_prob:.4f}")
print(f"Classification flipped: {original_prob >= threshold and new_prob < threshold}")


Before FGSM: P(fake) = 0.9145
After FGSM:  P(fake) = 0.0099
Classification flipped: True


In [7]:
# Map perturbed embeddings back to nearest real words
embedding_weights = model.embedding.weight.detach()  # (vocab_size, 100)
perturbed_squeezed = perturbed.squeeze(0).detach()    # (seq_len, 100)

# For each token position, find the closest real word
original_tokens = test_texts_tok[sample_idx]
seq_len = min(len(original_tokens), MAX_LEN)

changes = []
for pos in range(seq_len):
    old_id = original_tokens[pos]
    old_word = id_to_word.get(old_id, "<unk>")
    
    # Find nearest word to the perturbed vector
    dists = torch.cdist(perturbed_squeezed[pos].unsqueeze(0), embedding_weights).squeeze(0)
    new_id = dists.argmin().item()
    new_word = id_to_word.get(new_id, "<unk>")
    
    if old_id != new_id:
        changes.append((pos, old_word, new_word))

# Show full before/after text
original_words = [id_to_word.get(t, "<unk>") for t in original_tokens[:seq_len]]

# Build the perturbed sentence
perturbed_words = original_words.copy()
for pos, old_word, new_word in changes:
    perturbed_words[pos] = new_word

print("ORIGINAL TEXT:")
print(' '.join(original_words))
print(f"\nP(fake) = {original_prob:.4f}")

print("\nPERTURBED TEXT:")
print(' '.join(perturbed_words))
print(f"\nP(fake) = {new_prob:.4f}")


ORIGINAL TEXT:
nurse shift program supervisor us ky lexington apply using below linklink nurse shift program supervisor on one of five nursing units at thomson hood veterans center , a state owned long term , 285 bed , nursing care facility for kentucky s veterans and is located in jessamine county . you may contact <unk> <unk> at 1 link submit applications online at link apply to req <unk> br thomson hood veterans center does not discriminate against any person on the basis of race , color , national origin , disability , or age in admission , treatment , or participation in its programs , services , and activities , or in employment . for further information about this policy , contact amy lambert , telephone number tdd full time hospital health care

P(fake) = 0.9145

PERTURBED TEXT:
outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outbound outb

In [8]:
# Re-encode the perturbed words and run through the full model
perturbed_token_ids = [vocab.get(w, 0) for w in perturbed_words]
pad_len = MAX_LEN - len(perturbed_token_ids)
perturbed_padded = perturbed_token_ids + [1] * pad_len
perturbed_mask = [1] * len(perturbed_token_ids) + [0] * pad_len

perturbed_ids_tensor = torch.tensor([perturbed_padded], dtype=torch.long).to(device)
perturbed_mask_tensor = torch.tensor([perturbed_mask], dtype=torch.long).to(device)

model.eval()
with torch.no_grad():
    logit = model(perturbed_ids_tensor, perturbed_mask_tensor, numerical_features)
    word_swap_prob = torch.sigmoid(logit).item()

print(f"Original text:        P(fake) = {original_prob:.4f}")
print(f"Perturbed embeddings: P(fake) = {new_prob:.4f}")
print(f"Word-swapped text:    P(fake) = {word_swap_prob:.4f}")
print(f"\nWord swap fools model: {word_swap_prob < threshold}")


Original text:        P(fake) = 0.9145
Perturbed embeddings: P(fake) = 0.0099
Word-swapped text:    P(fake) = 0.0382

Word swap fools model: True
